# Introducción a Polars

[Polars](https://pola.rs/) es una librería de Python orientada a la **manipulación y análisis de datos tabulares**. 
Su API está pensada para trabajar de manera clara y eficiente con tablas, selecciones, filtros, agregaciones y transformaciones de columnas.

Las dos estructuras más importantes son:

* `DataFrame`: una tabla bidimensional con filas y columnas.
* `Series`: una estructura unidimensional que representa una columna de la tabla.

In [ ]:
import polars as pl

import pyprojroot

ROOT = pyprojroot.here()

A lo largo de este apunte vamos a trabajar con datos de los juegos olímpicos que se encuentran en `olympics.csv`.
Este conjunto de datos, que contiene datos básicos de los atletas que participaron en los juegos olímpicos entre Atenas 1896 y Río 2016, fue publicado en la semana 32 del año 2024 proyecto TidyTuesday y contiene las siguientes variables:

| Columna  | Descripción                                                              |
|----------|--------------------------------------------------------------------------|
| `id`     | Identificador del atleta.                                                |
| `name`   | Nombre del atleta.                                                       |
| `sex`    | Sexo del atleta.                                                         |
| `age`    | Edad del atleta.                                                         |
| `height` | Altura del atleta en centímetros.                                        |
| `weight` | Peso del atleta en kilogramos.                                           |
| `team`   | País o equipo por el que compite.                                        |
| `noc`    | Código NOC asociado a la región o delegación.                            |
| `games`  | Nombre de la edición de los Juegos Olímpicos.                            |
| `year`   | Año de los Juegos Olímpicos.                                             |
| `season` | Temporada de los Juegos: invierno o verano.                              |
| `city`   | Ciudad anfitriona de los Juegos Olímpicos.                               |
| `sport`  | Deporte.                                                                 |
| `event`  | Prueba o evento específico.                                              |
| `medal`  | Medalla obtenida: oro, plata, bronce o valor faltante si no obtuvo medalla. |

## Lectura de datos tabulares

La lectura de datos es una de las tareas fundamentales en el análisis de datos. En Polars, muchas fuciones para importar datos comienzan con `read_` y devuelven un `DataFrame`.
Algunos ejemplos son:

* `pl.read_csv()`
* `pl.read_parquet()`
* `pl.read_excel()`
* `pl.read_json()`
* `pl.read_database()`

Los archivos CSV (valores separados por comas) son uno de los formatos más comunes para almacenar datos tabulares. En este caso, los datos están en `olympics.csv`. Vamos a usar `pl.read_csv()` para leerlos.

In [ ]:
datos = pl.read_csv(ROOT / "datos" / "olympics.csv")

## Inspección de datos tabulares

La representación de un `DataFrame` en una Jupyter Notebook permite echar un primer vistazo a los datos:

In [ ]:
datos

No solo es posible ver las primeras y últimas filas de la tabla, sino que también es posible ver la cantidad de filas y columnas (`shape: (271_116, 15)`) y el tipo de datos de cada una de las variables. Con el objetivo de hacer un uso eficiente del almacenamiento y poder de procesamiento disponible, Polars ofrece una gran variedad de tipos de datos. A lo largo del curso los iremos conociendo.

In [ ]:
type(datos)

Si quisieramos ver solamente las primeras filas o columnas, podemos utilizar los métodos `.head()` y `.tail()` de `DataFrame`. Estos cuentan con un parámetro opcional `n` que permite controlar la cantidad de filas obtenidas.

In [ ]:
datos.head()

In [ ]:
datos.tail()

Para inspeccionar la estructura del `DataFrame` de manera programática se puede consultar su esquema (`schema`), los tipos de datos (`dtypes`) o la la cantidad de filas y columnas (`shape`).

También es posible obtener una tabla con el conteo de valores nulos por columna usando el método `null_count`.

En el caso de necesitar un resumen rápido y más completo, podemos hacer uso del método `describe`.

Si se quieren descartar rápidamente filas con valores nulos en algunas de sus columnas, se usa el método `drop_nulls`.

Vale la pena notar que los `DataFrame` también cuentan con el método `drop_nans`, pero este descarta filas con valores numéricos que no son válidos.

## Cálculo de resumenes

A la hora de trabajar con datos, suele ser de interés procesarlos de alguna manera para obtener información respecto de algunas variables relevantes. 
Por ejemplo, se pueden calcular valores medios, mínimos, máximos, medianas, diferencias, razones, etc. 
Las cuentas que resumen muchos datos en un único valor se llaman resumenes o agregaciones (porque se están resumiendo agregando varios valores de alguna forma en un único valor).

### Resumenes simples

Como ya vimos, una forma muy rápida de calcular estadísticas básicas con un `DataFrame` es utilizar el método `.describe`, que genera estadísticas básicas para las columnas numéricas del conjunto de datos. Veamos que también nos permite especificar el listado de percentiles que queremos utilizar.

A nivel columna, si queremos calcular la cantidad de ocurrencias de una columna categórica o discreta, podemos seleccionar la columna y luego usar `value_counts()`.

Notemos que el método `value_counts` pertenece a la `Series` y no al `DataFrame`.

Este método también recibe los parámetros `sort` que ordena las filas de mayor a menor cantidad de ocurrencias, `name` que permite renombrar la columna de conteo y `normalize` que divide los conteos por su suma de modo tal que representan una proporción.

In [ ]:
datos["noc"].value_counts(sort=True, name="p", normalize=True)

Otros métodos interesantes son `unique` y `.n_unique`, que devuelven los valores únicos de una columna y su cantidad.

In [ ]:
datos["noc"].unique()

In [ ]:
datos["noc"].n_unique()

In [ ]:
datos["games"].unique()

En cuanto a las variables numéricas, las `Series` de Polars ofrecen una gran variedad de resumenes estadísticos:

In [ ]:
datos["age"].mean()

In [ ]:
print("Media:", datos["age"].mean())
print("Mediana:", datos["age"].median())
print("Desvío estándar:", datos["age"].std())
print("Mínimo:", datos["age"].min())
print("Máximo:", datos["age"].max())
print("Percentil 10:", datos["age"].quantile(0.1))


### Expresiones

En Polars, es posible construir expresiones que representan transformaciones sobre los datos.

Para hacerlo, primero se indican las columnas con `pl.col()` y luego se las combina usando operadores aritméticos habituales o métodos provistos por Polars.

Por ejemplo, la siguiente expresión combina las columnas `"height"` y `"weight"` de modo tal que se calcule el índice de masa corporal.


In [ ]:
pl.col("weight") / (pl.col("height") / 100) ** 2

Como se puede ver, el bloque anterior todavía no realiza ningún cálculo sobre el conjunto de datos. Lo que hace es definir, de forma abstracta, una transformación que queremos aplicar.

Para evaluar esa expresión, en este caso para calcular el índice de masa corporal, necesitamos usarla dentro de un contexto de ejecución. Polars ofrece varios, pero por ahora nos interesa `select`. Este contexto permite aplicar expresiones sobre las columnas y producir nuevas columnas, ya sea como resúmenes, combinaciones de columnas existentes o incluso valores literales.

Veamos que, en principio, `select` permite seleccionar columnas:

Sin embargo, como acabamos de mencionar, también permite ejecutar expresiones.

In [ ]:
datos.select("name", pl.col("age") * 12)

Si queremos guardar el resultado de la expresión en una nueva columna, tenemos que pasarla como parámetro:

In [ ]:
datos.select(
    "name", 
    "height", 
    "weight", 
    bmi=pl.col("weight") / (pl.col("height") / 100) ** 2
)

O utilizar el método `alias`.

Retemando el cálculo de resumenes, es posible utilizar un método de agregación, por ejempĺo:

In [ ]:
datos.select(pl.col("age").mean())

O bien:

In [ ]:
datos.select(
    pl.col("age").mean().alias("edad_media"),
    pl.col("age").std().alias("edad_std"),
)

Como iremos viendo a lo largo del curso, las expresiones de Polars son muy potentes y se pueden usar en varios contextos, no solo en `select`.

### Resumenes agrupados

El siguiente paso es calcular resumenes para cada nivel de otra variable (o combinaciones de niveles de múltiples variables).

Por ejemplo:

* ¿Cuál es la edad promedio de los atletas en cada juego olímpico?
* ¿Cuántas medallas de oro ganó cada país en cada edición?
* ¿Qué país estuvo más represenado por mujeres?

Lo primero que tenemos que conocer para realizar esta tarea es método `group_by`, que es un contexto de ejecución de expresiones.

Este método agrupa el `DataFrame` según una o más columnas para que luego podamos aplicar agregaciones sobre cada grupo.

In [ ]:
datos.group_by("year")

La representación del objeto agrupado es bastante austera, pero una vez agrupados los datos podemos aplicar agregaciones con `.agg()`.

In [ ]:
datos.group_by("games").agg(pl.col("age").mean())

Utilizando el método `sort()` se puede obtener una tabla ordenada según variables de interés, por ejemplo `games`.

Y con un poco de creatividad podemos obtener la edición donde se presentó el atleta mas longevo:

Otro ejemplo:

In [ ]:
(
    datos
    .group_by("games", "noc")
    .agg(
        (pl.col("sex") == "F").mean().alias("p_mujer"),
        pl.len().alias("n")
    )
    .sort("p_mujer", "n")
    .tail(10)
)

Una forma de obtener la cantidad de observaciones por grupo es:

In [ ]:
datos.group_by("games", "noc").agg(pl.len())

## Manipulación de columnas

### Seleccionar y transformar

Podemos seleccionar una columna de un `DataFrame` pasando el nombre de la misma dentro de corchetes (de la misma forma que haríamos con un diccionario).

El resultado es una `Series`, no un `DataFrame` compuesto por una sola columna.

Si queremos obtener un `DataFrame`, tenemos que usar el método `select`.

In [ ]:
# con alias

In [ ]:
 # columna y resumen

In [ ]:
# estandarizar peso

Si se desea modificar y/o agregar columnas a un `DataFrame`, conviene utilizar el contexto `with_columns`, que devuelve una tabla con todas las columnas del `DataFrame` original junto con las nuevas columnas generadas a partir de las expresiones.

In [ ]:
datos.with_columns(
    weight_z=(pl.col("weight") - pl.col("weight").mean()) / pl.col("weight").std()
)

Debido a esta diferencia entre `select` y with_co`lumns, las expresiones usadas dentro de `with_columns` deben producir series que tengan la misma longitud que las columnas originales del `DataFrame`. 
En cambio, en el contexto `select` alcanza con que las expresiones produzcan series de la misma longitud entre sí.

### Descartar

Para descartar columnas tenemos el método `.drop`. Simplemente le pasamos el nombre de las columnas que queremos descartar.

In [ ]:
datos.drop("id", "year", "season")

### Renombrar

In [ ]:
datos.select("name", "sex", "age").rename({"name": "nombre", "sex": "sexo", "age": "edad"})

## Manipulación de filas

### Ordenar

Para ordenar las observaciones en función de una o más variables usamos `.sort()`. Si pasamos una sola columna alcanza con una cadena; si queremos ordenar por varias, conviene pasar una lista.

El argumento `descending` permite invertir el orden. Como Polars no usa un índice persistente, ordenar las filas no requiere ningún paso extra para reenumerarlas.

In [ ]:
datos.sort("year")

In [ ]:
datos.sort("year", descending=True)

In [ ]:
datos.sort(["season", "year"])

### Filtrar

Es posible seleccionar filas por posición usando _slices_.  Esto resulta útil para inspeccionar rápidamente una parte de la tabla.

In [ ]:
datos[0:10]

In [ ]:
datos[0:10:2]

In [ ]:
datos[0:100:10, :5]

Cuando lo que queremos es conservar filas según una condición sobre sus valores, en cambio, conviene usar el método `filter`.
Este método acepta condiciones en forma de expresiones booleanas, es decir, `filter` también es un contexto.

Por ejemplo, podemos quedarnos solo con los Juegos Olímpicos de invierno, excluirlos o combinar condiciones sobre `season`, `year`, `sport` o `medal`.

In [ ]:
# Invierno

In [ ]:
# No invierno

In [ ]:
# Verano a partir del 2000

In [ ]:
# Atletas que ganaron oro o plata

In [ ]:
# Deporte Athletics y Swimming entre 2000 y 2016


In [ ]:
# Antes a 1950 o luego de 2010

Cuando usamos operadores como `|` conviene encerrar cada comparación entre paréntesis para que la expresión se evalúe como esperamos.

### Descartar duplicados

En Polars podemos usar `.unique()` para quedarnos con combinaciones únicas de una o más columnas. Esto sirve tanto para explorar valores posibles como para descartar filas duplicadas según un subconjunto de variables.

Si usamos el parámetro `subset`, Polars decide si dos filas son duplicadas mirando solo esas columnas. 

In [ ]:
datos.select("season", "sport").unique()

In [ ]:
datos.unique(subset=["season", "sport"])

## Combinación de tablas

Para combinar tablas en Polars hay dos operaciones muy comunes: la **concatenación** y la **fusión**. La concatenación pega tablas por posición, mientras que la fusión relaciona filas a partir de una o más claves.

### Concatenación

La concatenación combina tablas que ya tienen una estructura compatible. Con `pl.concat()` podemos apilarlas verticalmente o pegarlas horizontalmente.

En una concatenación vertical, las filas de una tabla se agregan debajo de las de otra. En una concatenación horizontal, en cambio, las columnas se agregan lado a lado y las filas se alinean por posición.

In [ ]:
ventas_2023 = pl.DataFrame({
    "id_venta": [1, 2, 3],
    "producto": ["Cuaderno", "Lapicera", "Regla"],
    "unidades": [5, 8, 3],
})

ventas_2024 = pl.DataFrame({
    "id_venta": [4, 5],
    "producto": ["Cuaderno", "Mochila"],
    "unidades": [6, 2],
})

detalle_ventas = pl.DataFrame({
    "id_venta": [1, 2, 3],
    "producto": ["Cuaderno", "Lapicera", "Regla"],
})

precios = pl.DataFrame({
    "precio_unitario": [1200, 800, 1500],
    "descuento": [0.10, 0.00, 0.05],
})

In [ ]:
pl.concat([ventas_2023, ventas_2024], how="vertical")

In [ ]:
pl.concat([detalle_ventas, precios], how="horizontal")

### Fusión

Cuando queremos combinar tablas a partir de una clave compartida, usamos `.join()`. Los tipos de fusión más comunes son `inner`, `left`, `right` y `full`.

* `inner`: conserva solo las filas con coincidencia en ambas tablas.
* `left`: conserva todas las filas de la tabla izquierda y agrega coincidencias de la derecha.
* `right`: conserva todas las filas de la tabla derecha y agrega coincidencias de la izquierda.
* `full`: conserva todas las filas de ambas tablas, aunque no tengan coincidencia.

La documentación oficial de Polars muestra más variantes y detalles sobre joins: https://docs.pola.rs/user-guide/transformations/joins/

In [ ]:
clientes = pl.DataFrame({
    "id_cliente": [101, 102, 103, 105],
    "nombre": ["Ana", "Bruno", "Carla", "Diego"],
})

segmentos = pl.DataFrame({
    "id_cliente": [102, 103, 104],
    "segmento": ["Mayorista", "Minorista", "Mayorista"],
})

In [ ]:
clientes.join(segmentos, on="id_cliente", how="inner")

In [ ]:
clientes.join(segmentos, on="id_cliente")

El resultado es el mismo porque `how` es por defecto igual a `"inner"`.

In [ ]:
clientes.join(segmentos, on="id_cliente", how="left")

In [ ]:
clientes.join(segmentos, on="id_cliente", how="right")

In [ ]:
clientes.join(segmentos, on="id_cliente", how="full")

También es posible unir usando más de una columna. En ese caso, le pasamos a `on` una lista con los nombres de las claves que deben coincidir simultáneamente.

In [ ]:
ventas_sucursal = pl.DataFrame({
    "sucursal": ["Centro", "Centro", "Norte", "Norte"],
    "producto": ["Cuaderno", "Lapicera", "Cuaderno", "Mochila"],
    "unidades": [20, 35, 18, 7],
})

precios_sucursal = pl.DataFrame({
    "sucursal": ["Centro", "Centro", "Norte", "Sur"],
    "producto": ["Cuaderno", "Lapicera", "Cuaderno", "Mochila"],
    "precio_unitario": [1200, 800, 1250, 32000],
})

ventas_sucursal.join(precios_sucursal, on=["sucursal", "producto"], how="left")

## Escritura de de datos

Así como Polars ofrece varias funciones para leer datos, también incluye métodos para exportarlos. Uno de los más usados es `.write_csv`, aunque también existen `.write_parquet` y `.write_excel` (estos últimos pueden requerir dependencias adicionales).

El primer argumento de `.write_csv` es la ruta donde queremos guardar el archivo. 

In [ ]:
datos.head(10).write_csv("primeras_10_filas.csv")

In [ ]:
datos

In [ ]:
df_equipos = (
    datos
    .filter(pl.col("season") == "Summer")
    .group_by("year")
    .agg(pl.col("noc").n_unique())
    .sort("year")
)
df_equipos

In [ ]:
df_equipos.write_csv("equipos_por_año.csv")

## Enlaces útiles

- [Python Polars: The Definitive Guide (Libro)](https://polarsguide.com/)
- [Guía oficial de Polars](https://docs.pola.rs/])
- [Polars Cheat Sheet](https://franzdiebold.github.io/polars-cheat-sheet/Polars_cheat_sheet.pdf)